# Pipeline smoke test

Runs steps 1–3 of `plan.md` end-to-end:
1. Load Football-Data.co.uk results + B365 odds
2. Load Understat per-match team xG
3. Join on canonical team names + date
4. Build the lag-safe rolling 5-game xG-diff feature

Re-runs are cached on disk under `data/raw/`.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is opened from the notebooks/ folder.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Football-Data.co.uk — results + bookmaker odds

In [ ]:
from src.load import load_football_data

fd = load_football_data("2022-2023")
print(fd.shape)
fd.head()

## 2. Understat — per-match team xG

In [ ]:
from src.load import load_team_xg

xg = load_team_xg("2022-2023")
print(xg.shape)
xg.head()

## 3. Join sources (asserts zero unmatched rows)

In [ ]:
from src.join import join_sources

joined = join_sources(fd, xg)
print(joined.shape)
joined.head()

## 4. Lag-safe rolling 5-game xG-diff feature

In [ ]:
from src.features import rolling_xg_diff

feat = rolling_xg_diff(joined, xg, window=5)
print("non-null xg_diff_5:", feat["xg_diff_5"].notna().sum(), "/", len(feat))
feat[["match_id", "date", "home_team", "away_team", "xg_diff_5"]].head(10)

In [ ]:
feat["xg_diff_5"].describe()

## Sanity check — top end-of-season form

In [ ]:
(feat.dropna(subset=["xg_diff_5"])
      .sort_values("date")
      .tail(20)[["date", "home_team", "away_team", "result", "xg_diff_5"]])